# Retail Sales Analysis and AI-Based Sales Prediction

**Internship:** AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026 — BharatCares  
**Level:** Beginner  
**Note:** This notebook starts with a small illustrative dataset generated inside the notebook, so it runs without downloading a dataset. Replace it with an approved real dataset before making real-world business claims.

## Project objectives
- Explore sales, quantity, product category, and region.
- Summarize sales performance using pandas.
- Visualize trends using Matplotlib and Seaborn.
- Train a simple Linear Regression model to estimate sales from advertising spend and units sold.
- Evaluate the model using MAE and R², and explain its limitations.

In [ ]:
# If you need to install dependencies in a local environment, run:
# %pip install -r requirements.txt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
print('Libraries imported successfully.')

## 1. Create or load the dataset

The sample data is synthetic and intended only to demonstrate the workflow. It is generated with a fixed random seed so results are reproducible.

In [ ]:
rng = np.random.default_rng(42)
n_rows = 180

dates = pd.date_range(start='2025-01-01', periods=n_rows, freq='D')
categories = np.array(['Electronics', 'Home', 'Clothing', 'Books'])
regions = np.array(['North', 'South', 'East', 'West'])

units_sold = rng.integers(1, 25, size=n_rows)
advertising_spend = rng.uniform(100, 2000, size=n_rows).round(2)
unit_price = rng.uniform(10, 250, size=n_rows).round(2)
category = rng.choice(categories, size=n_rows)
region = rng.choice(regions, size=n_rows)
noise = rng.normal(0, 120, size=n_rows)
sales = (units_sold * unit_price + 0.8 * advertising_spend + noise).clip(min=20).round(2)

df = pd.DataFrame({
    'Date': dates,
    'Category': category,
    'Region': region,
    'Units_Sold': units_sold,
    'Unit_Price': unit_price,
    'Advertising_Spend': advertising_spend,
    'Sales': sales
})

# Optional: To use your own CSV, uncomment and edit the next line.
# df = pd.read_csv('your_sales_data.csv', parse_dates=['Date'])

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
display(df.head())

## 2. Understand and clean the data

Check data types, missing values, and duplicate records before analysis.

In [ ]:
print('Data types and non-null counts:')
df.info()

print('\nMissing values per column:')
display(df.isna().sum().to_frame('Missing values'))

print('Duplicate rows:', df.duplicated().sum())

# Basic cleaning steps: remove exact duplicates and rows missing key analysis values.
df = df.drop_duplicates().copy()
required_columns = ['Units_Sold', 'Advertising_Spend', 'Sales']
df = df.dropna(subset=required_columns)
print('Rows after basic cleaning:', len(df))

## 3. Exploratory data analysis (EDA)

Use descriptive statistics and grouped summaries to understand the sample data.

In [ ]:
print('Numerical summary:')
display(df[['Units_Sold', 'Unit_Price', 'Advertising_Spend', 'Sales']].describe().round(2))

print('Total sales by category:')
category_sales = df.groupby('Category', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False)
display(category_sales.round(2))

print('Total sales by region:')
region_sales = df.groupby('Region', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False)
display(region_sales.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=category_sales, x='Category', y='Sales', ax=axes[0])
axes[0].set_title('Total Sales by Product Category')
axes[0].set_ylabel('Sales (illustrative units)')
axes[0].tick_params(axis='x', rotation=25)

sns.barplot(data=region_sales, x='Region', y='Sales', ax=axes[1])
axes[1].set_title('Total Sales by Region')
axes[1].set_ylabel('Sales (illustrative units)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Advertising_Spend', y='Sales', hue='Category', alpha=0.8)
plt.title('Advertising Spend vs Sales')
plt.xlabel('Advertising spend (illustrative units)')
plt.ylabel('Sales (illustrative units)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.heatmap(df[['Units_Sold', 'Unit_Price', 'Advertising_Spend', 'Sales']].corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Between Numerical Variables')
plt.tight_layout()
plt.show()

## 4. AI / Machine Learning: estimate sales

A Linear Regression model learns a simple relationship between `Units_Sold`, `Unit_Price`, and `Advertising_Spend` and the target `Sales`. This is a beginner baseline, not a production forecasting system.

In [ ]:
features = ['Units_Sold', 'Unit_Price', 'Advertising_Spend']
target = 'Sales'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f'Mean Absolute Error (MAE): {mae:.2f}')
print(f'R-squared (R²): {r2:.3f}')
print('\nModel coefficients:')
display(pd.DataFrame({'Feature': features, 'Coefficient': model.coef_.round(3)}))
print(f'Intercept: {model.intercept_:.2f}')

In [ ]:
comparison = pd.DataFrame({'Actual_Sales': y_test.values, 'Predicted_Sales': predictions.round(2)})
display(comparison.head(10))

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=predictions)
min_value = min(y_test.min(), predictions.min())
max_value = max(y_test.max(), predictions.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle='--')
plt.xlabel('Actual sales')
plt.ylabel('Predicted sales')
plt.title('Actual vs Predicted Sales')
plt.tight_layout()
plt.show()

## 5. Try a prediction

Change the example values below to estimate sales for a hypothetical transaction. Use the same units as the training data.

In [ ]:
example = pd.DataFrame({
    'Units_Sold': [10],
    'Unit_Price': [75.0],
    'Advertising_Spend': [500.0]
})
estimated_sales = model.predict(example)[0]
print(f'Estimated sales for the example: {estimated_sales:.2f} illustrative units')

## 6. Findings, limitations, and conclusion

After running the notebook, use the category and region summary tables to identify the highest-total groups in this generated sample. Use the printed MAE and R² values to describe the model's test performance.

**Important limitations**
- The dataset is synthetic, not real company data.
- The relationship was intentionally created for demonstration, so results must not be presented as real retail findings.
- A random train/test split is suitable for this simple demonstration, but time-ordered data should usually be evaluated with a time-based split for genuine forecasting.
- Correlation and model coefficients do not prove causation.
- Before real deployment, validate data quality, compare with baseline models, check error patterns, and protect customer/business data.

**Conclusion:** This project demonstrates an end-to-end beginner workflow: data preparation, exploratory analysis, visualization, a basic machine-learning model, evaluation, and interpretation. Replace the sample data with a properly licensed dataset for a stronger final submission.